# CT: from X-ray attenuation to an inverse problem

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/guanhuaw/MIRTorch/blob/master/examples/demo_ct.ipynb)

This self-contained tutorial connects **2D parallel-/fan-beam geometry**, **Beer–Lambert photon counts**, and **nonnegative iterative reconstruction**. We first check physical path lengths and gradients, then reconstruct a synthetic attenuation image with the same solver used elsewhere in MIRTorch.

A matched adjoint is necessary for reconstruction, but is not evidence that a scanner model is complete. This example uses monochromatic pencil rays and known calibration; it does not model cone beams, scatter transport, beam hardening, or finite detector/focal-spot widths.

In [ ]:
# In Colab, clone MIRTorch and install its example dependencies.
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules or "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    repository = Path("/content/MIRTorch")
    if not (repository / "pyproject.toml").exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "https://github.com/guanhuaw/MIRTorch.git",
                str(repository),
            ],
            check=True,
        )
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--editable",
            f"{repository}[examples]",
        ],
        check=True,
    )
    os.chdir(repository)

In [ ]:
import math
import time

import matplotlib.pyplot as plt
import torch

from mirtorch.alg import FISTA
from mirtorch.linear import CT, Diff2dgram
from mirtorch.prox import BoxConstraint
from mirtorch.util import l2_norm


def best_available_device():
    override = os.environ.get("MIRTORCH_EXAMPLE_DEVICE")
    if override:
        return torch.device(override)
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


torch.manual_seed(0)
device = best_available_device()
print(f"Using {device} with PyTorch {torch.__version__}")

## 1. A line integral is not a photon count

For attenuation $\mu_j$ in cm$^{-1}$ and ray–pixel path length $a_{ij}$ in cm,

$$p_i = (A\mu)_i = \sum_j a_{ij}\mu_j,\qquad
\bar y_i = I_{0,i}\exp(-p_i) + b_i,\qquad y_i\sim\mathrm{Poisson}(\bar y_i).$$

`A(mu)` is linear; `A.expected_counts(mu, incident, background)` predicts the nonlinear Poisson mean. The incident count includes source/detector calibration, and the background is a supplied additive count estimate. Raw Hounsfield units cannot be passed as attenuation coefficients.

Our image is `(rows=y, columns=x)`, with both physical coordinates increasing with index. At angle zero rays travel along +y; detector bins run along +x. Angles are in **radians**. Fan rays connect a point source to a flat detector, so their paths and magnification differ from parallel rays. Geometry is fixed when the operator is built; image and count-calibration parameters are differentiable.

In [ ]:
size = 64
pixel_size = 0.375  # cm; a 24 cm square field of view.
coordinate = (torch.arange(size, device=device) - (size - 1) / 2) * pixel_size
y, x = torch.meshgrid(coordinate, coordinate, indexing="ij")
body = (x / 8.5).square() + (y / 10.0).square() <= 1
bone = ((x + 2.8) / 1.6).square() + ((y - 1.5) / 3.5).square() <= 1
air = ((x - 2.5) / 2.3).square() + ((y + 1.5) / 4.0).square() <= 1
mu_true = (0.16 * body + 0.09 * bone - 0.12 * air).clamp_min(0)

parallel = CT(
    mu_true.shape,
    torch.arange(96, device=device) * math.pi / 96,
    96,
    voxel_size=pixel_size,
    detector_spacing=0.4,
)
fan = CT(
    mu_true.shape,
    torch.arange(128, device=device) * (2 * math.pi / 128),
    128,
    voxel_size=pixel_size,
    detector_spacing=0.6,
    geometry="fan",
    source_distance=50.0,
    detector_distance=50.0,  # cm; central magnification = 2.
)
models = {"Parallel": parallel, "Fan": fan}

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].imshow(mu_true.cpu(), origin="lower", cmap="gray", vmin=0, vmax=0.25)
axes[0].set_title("Attenuation truth (cm$^{-1}$)")
for axis, (name, operator) in zip(axes[1:], models.items()):
    plot = axis.imshow(operator(mu_true).cpu(), origin="lower", aspect="auto")
    axis.set_title(f"{name}: line integrals")
    axis.set_xlabel("Detector bin")
    axis.set_ylabel("View")
    fig.colorbar(plot, ax=axis, shrink=0.8)
plt.tight_layout()

## 2. Check physics and derivatives before reconstructing

For a uniform rectangle, the center ray at angle zero must integrate the full physical height. This is an independent analytic check, not a comparison of the projector with its own adjoint. We also check $\langle A x,q\rangle=\langle x,A^Hq\rangle$ and the count-model gradient

$$\nabla_\mu \langle q,\bar y\rangle = -A^H\!\left(q\odot I_0\exp(-A\mu)\right).$$

The test suite additionally compares individual ray–pixel intersections to an independent clipping implementation and verifies convergence to analytic disk chord lengths as the grid is refined.

In [ ]:
rectangle = CT((8, 10), torch.tensor([0.0], device=device), 11, voxel_size=(0.3, 0.2))
center_integral = rectangle(torch.full((8, 10), 0.15, device=device))[0, 5]
torch.testing.assert_close(center_integral, torch.tensor(8 * 0.3 * 0.15, device=device))
print(
    f"Uniform rectangle: computed {center_integral.item():.6f}, analytic {8 * 0.3 * 0.15:.6f}"
)

for name, operator in models.items():
    image = torch.randn_like(mu_true)
    probe = torch.randn(operator.size_out, device=device)
    lhs = (operator(image) * probe).sum()
    rhs = (image * operator.H(probe)).sum()
    relative_error = (lhs - rhs).abs() / (lhs.abs() + rhs.abs()).clamp_min(1e-6)
    assert relative_error < 2e-5
    print(f"{name} adjoint relative error: {relative_error.item():.2e}")

mu = mu_true.clone().requires_grad_()
probe = torch.randn(fan.size_out, device=device)
mean_counts = fan.expected_counts(mu, incident=20_000.0, background=5.0)
actual_gradient = torch.autograd.grad((mean_counts * probe).sum(), mu)[0]
expected_gradient = -fan.H(probe * (mean_counts.detach() - 5.0))
torch.testing.assert_close(actual_gradient, expected_gradient, rtol=3e-5, atol=0.1)
print("Beer–Lambert gradient agrees with the analytic adjoint expression.")

## 3. Noisy transmission data and a constrained inverse problem

We reconstruct with a quadratic roughness penalty and nonnegative attenuation:

$$\hat\mu = \arg\min_{\mu\ge0}\frac12\|W^{1/2}(A\mu-p)\|_2^2 + \frac\beta2\|D\mu\|_2^2.$$

Here $p=-\log[(y-b)/I_0]$ and $W$ uses a delta-method variance approximation. The lower clamp only protects the logarithm against nonpositive background-subtracted counts. This approximation is suitable for this high-count example; at low counts, use the Poisson likelihood directly rather than treating the logarithm as unbiased Gaussian data.

FISTA uses a conservative step-size bound, not a guessed learning rate. The normalized backprojection is only a baseline, not an analytic inverse. Both geometries are reconstructed separately.

In [ ]:
incident, background = 20_000.0, 5.0
beta = 0.2
roughness = Diff2dgram(mu_true.shape, compile=False)


def reconstruct(operator):
    # Draw on CPU: some Metal installations do not have Poisson sampling kernels.
    mean = operator.expected_counts(mu_true, incident, background)
    counts = torch.poisson(mean.cpu()).to(device)
    transmitted = (counts - background).clamp_min(1.0)
    log_data = -torch.log(transmitted / incident)
    weights = transmitted.square() / counts.clamp_min(1.0) / incident

    def objective(value):
        residual = operator(value) - log_data
        data_fit = 0.5 * (weights * residual.square()).sum()
        penalty = 0.5 * beta * (value * roughness(value)).sum()
        return data_fit + penalty

    def gradient(value):
        return operator.H(weights * (operator(value) - log_data)) + beta * roughness(
            value
        )

    # For nonnegative A,W, max row sum bounds ||A^H W A||_2; ||D^H D||_2 <= 8.
    row_sums = operator.H(weights * operator(torch.ones_like(mu_true)))
    lipschitz = row_sums.amax().item() + 8 * beta
    baseline = (operator.H(weights * log_data) / row_sums.clamp_min(1e-6)).clamp_min(0)
    solver = FISTA(
        gradient,
        lipschitz,
        BoxConstraint(1.0, 0.0, float("inf")),
        max_iter=100,
        restart=True,
        compile=False,
        eval_func=lambda value: objective(value).detach().cpu().item(),
    )
    start = time.perf_counter()
    with torch.no_grad():
        reconstructed, history = solver.run(baseline)
    elapsed = time.perf_counter() - start
    initial_error = (l2_norm(baseline - mu_true) / l2_norm(mu_true)).item()
    final_error = (l2_norm(reconstructed - mu_true) / l2_norm(mu_true)).item()
    assert torch.isfinite(reconstructed).all() and reconstructed.min() >= 0
    assert final_error < initial_error and history[-1] < history[0]
    print(
        f"NRMSE {initial_error:.3f} → {final_error:.3f}; 100 FISTA steps in {elapsed:.1f} s"
    )
    return baseline, reconstructed, history


results = {}
for name, operator in models.items():
    print(name)
    results[name] = reconstruct(operator)

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(13, 6))
for row, (name, (baseline, reconstructed, history)) in enumerate(results.items()):
    for axis, image, title in zip(
        axes[row, :3],
        [mu_true, baseline, reconstructed],
        ["Truth", "Normalized backprojection", f"{name}: nonnegative FISTA"],
    ):
        axis.imshow(image.cpu(), origin="lower", cmap="gray", vmin=0, vmax=0.25)
        axis.set_title(title)
        axis.axis("off")
    axes[row, 3].semilogy(torch.tensor(history) / history[0])
    axes[row, 3].set_title("Relative objective")
    axes[row, 3].set_xlabel("Iteration")
    axes[row, 3].grid(alpha=0.3)
plt.tight_layout()

## What this example establishes—and what it does not

- Physical lengths, fan geometry, matched adjoints, and image/count gradients are checked. The line integrals are exact for the **piecewise-constant pixel model**, not for an arbitrary continuous object.
- The reconstruction uses the same model to simulate and invert data. Its good images demonstrate solver consistency, not validation against an independent scanner or Monte Carlo simulation.
- Each detector bin samples one ray at its center. Finite-bin intensity integration would require averaging **transmitted intensities**, not exponentiating an average line integral.
- The geometry is fixed; gradient-requiring angles are rejected explicitly. The cached intersection plan is bounded in memory (128 MiB by default); large plans are recomputed in ray chunks. This cache budget excludes autograd activations, which can retain all chunk weights during differentiation. This portable implementation prioritizes correctness, not clinical-scale CT throughput.
- CT estimates attenuation from transmitted photons. [The SPECT example](https://github.com/guanhuaw/MIRTorch/blob/master/examples/demo_mlem.ipynb) instead estimates emitted activity using a known attenuation map and a depth-dependent collimator response.

Try halving the incident counts, reducing the view count, or changing `beta`. Compare noise, edge bias, and objective values; keep the model and data-generation assumptions visible.

References: [Siddon (1985)](https://doi.org/10.1118/1.595715), [ASTRA geometry conventions](https://astra-toolbox.com/docs/geom2d.html), and [line versus strip/Joseph projectors](https://astra-toolbox.com/docs/proj2d.html).